# NTO AI Final — Advanced Memory-Safe Pipeline (GPU)

Этот ноутбук сделан как **advanced** версия с фокусом на:
- высокий recall кандидатов,
- стабильную память (без row-wise apply по миллионам строк),
- CatBoostRanker (GPU),
- строгий контракт сабмита `top-20 unique edition_id per user`.


In [ ]:

# CELL 1 — Setup
import os
import gc
import math
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostRanker, Pool
from implicit.als import AlternatingLeastSquares

warnings.filterwarnings("ignore")

SEED = 42
TOP_K = 20
CANDS_PER_USER = 220
ALS_FACTORS = 128
ALS_ITERS = 22

DATA_DIR_CANDIDATES = [
    "/kaggle/input/nto-ai",
    "/kaggle/input/datasets/artemnazemtsev/nto-ai",
    "./data",
]


def detect_data_dir(paths):
    for p in paths:
        if os.path.exists(os.path.join(p, "interactions.csv")):
            return p
    raise FileNotFoundError("interactions.csv not found")


def mem(df, name):
    mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"{name}: {len(df):,} rows | {mb:.2f} MB")


In [ ]:

# CELL 2 — Data loading with strict dtypes
DATA_DIR = detect_data_dir(DATA_DIR_CANDIDATES)
print("DATA_DIR:", DATA_DIR)

interactions = pd.read_csv(
    os.path.join(DATA_DIR, "interactions.csv"),
    parse_dates=["event_ts"],
    dtype={"user_id": "int64", "edition_id": "int64", "event_type": "int8", "rating": "float32"},
)
targets = pd.read_csv(os.path.join(DATA_DIR, "targets.csv"), dtype={"user_id": "int64"})
editions = pd.read_csv(os.path.join(DATA_DIR, "editions.csv"))
book_genres = pd.read_csv(os.path.join(DATA_DIR, "book_genres.csv"))
users_path = os.path.join(DATA_DIR, "users.csv")
users = pd.read_csv(users_path) if os.path.exists(users_path) else pd.DataFrame({"user_id": interactions.user_id.unique()})

for c in ["edition_id", "book_id", "author_id", "language_id", "publisher_id", "publication_year"]:
    if c in editions.columns:
        editions[c] = pd.to_numeric(editions[c], errors="coerce")

for c in ["book_id", "genre_id"]:
    if c in book_genres.columns:
        book_genres[c] = pd.to_numeric(book_genres[c], errors="coerce")

mem(interactions, "interactions")
mem(targets, "targets")
mem(editions, "editions")


In [ ]:

# CELL 3 — Time split + base features
T_MAX = interactions["event_ts"].max()
T_INCIDENT_START = pd.Timestamp("2025-10-01")
if pd.isna(T_INCIDENT_START) or T_INCIDENT_START < interactions["event_ts"].min() or T_INCIDENT_START > T_MAX:
    T_INCIDENT_START = T_MAX - pd.Timedelta(days=31)
T_TRAIN_END = T_INCIDENT_START - pd.Timedelta(days=31)

train_log = interactions[interactions["event_ts"] < T_TRAIN_END].copy()
valid_log = interactions[(interactions["event_ts"] >= T_TRAIN_END) & (interactions["event_ts"] < T_INCIDENT_START)].copy()

print("train_log:", len(train_log), "valid_log:", len(valid_log))

POS_EVENTS = [1, 2]
pos_train = train_log[train_log["event_type"].isin(POS_EVENTS)].copy()

item_pop = pos_train.groupby("edition_id").size().astype("float32").rename("edition_pop")
item_recent = pos_train[pos_train["event_ts"] >= (T_INCIDENT_START - pd.Timedelta(days=31))].groupby("edition_id").size().astype("float32").rename("edition_recent_pop")
item_users = pos_train.groupby("edition_id")["user_id"].nunique().astype("float32").rename("edition_unique_users")
item_read = pos_train[pos_train["event_type"].eq(2)].groupby("edition_id").size().astype("float32").rename("read_pop")
item_wish = pos_train[pos_train["event_type"].eq(1)].groupby("edition_id").size().astype("float32").rename("wish_pop")

user_act = pos_train.groupby("user_id").size().astype("float32").rename("user_activity")
user_uniq = pos_train.groupby("user_id")["edition_id"].nunique().astype("float32").rename("user_uniq_editions")

item_features = editions[["edition_id", "book_id", "author_id", "publication_year", "language_id", "publisher_id"]].copy()
item_features = item_features.merge(item_pop.reset_index(), on="edition_id", how="left")
item_features = item_features.merge(item_recent.reset_index(), on="edition_id", how="left")
item_features = item_features.merge(item_users.reset_index(), on="edition_id", how="left")
item_features = item_features.merge(item_read.reset_index(), on="edition_id", how="left")
item_features = item_features.merge(item_wish.reset_index(), on="edition_id", how="left")

for c in ["edition_pop", "edition_recent_pop", "edition_unique_users", "read_pop", "wish_pop"]:
    item_features[c] = item_features[c].fillna(0).astype("float32")
item_features["pop_log"] = np.log1p(item_features["edition_pop"]).astype("float32")
item_features["recent_pop_log"] = np.log1p(item_features["edition_recent_pop"]).astype("float32")
item_features["read_ratio"] = (item_features["read_pop"] / (item_features["edition_pop"] + 1)).astype("float32")
item_features["wish_ratio"] = (item_features["wish_pop"] / (item_features["edition_pop"] + 1)).astype("float32")

user_features = pd.DataFrame({"user_id": pd.Index(np.union1d(users["user_id"].unique(), targets["user_id"].unique()))})
user_features = user_features.merge(user_act.reset_index(), on="user_id", how="left")
user_features = user_features.merge(user_uniq.reset_index(), on="user_id", how="left")
for c in ["user_activity", "user_uniq_editions"]:
    user_features[c] = user_features[c].fillna(0).astype("float32")
user_features["user_activity_log"] = np.log1p(user_features["user_activity"]).astype("float32")

if "gender" in users.columns:
    user_features = user_features.merge(users[["user_id", "gender"]], on="user_id", how="left")
if "age" in users.columns:
    user_features = user_features.merge(users[["user_id", "age"]], on="user_id", how="left")
for c in ["gender", "age"]:
    if c in user_features.columns:
        user_features[c] = pd.to_numeric(user_features[c], errors="coerce").fillna(0).astype("float32")


In [ ]:

# CELL 4 — ALS (GPU) with time-decay weights
pos = train_log[train_log["event_type"].isin(POS_EVENTS)].copy()
pos["base_w"] = pos["event_type"].map({1: 1.0, 2: 2.0}).astype("float32")
max_ts = pos["event_ts"].max()
pos["days_old"] = (max_ts - pos["event_ts"]).dt.days.clip(lower=0)
pos["w"] = (pos["base_w"] * np.exp(-np.log(2) * pos["days_old"] / 30)).astype("float32")

u_enc = LabelEncoder()
i_enc = LabelEncoder()
u_idx = u_enc.fit_transform(pos["user_id"])
i_idx = i_enc.fit_transform(pos["edition_id"])

X = csr_matrix((pos["w"].to_numpy(), (u_idx, i_idx)), shape=(len(u_enc.classes_), len(i_enc.classes_)))

als = AlternatingLeastSquares(
    factors=ALS_FACTORS,
    iterations=ALS_ITERS,
    regularization=0.02,
    random_state=SEED,
    use_gpu=True,
)
als.fit(X)

u2als = {u: j for j, u in enumerate(u_enc.classes_)}
e2als = {e: j for j, e in enumerate(i_enc.classes_)}
als_items = i_enc.classes_

seen_pairs = set(zip(pos["user_id"].to_numpy(), pos["edition_id"].to_numpy()))
user_hist = pos.sort_values("event_ts").groupby("user_id")["edition_id"].apply(lambda s: list(s)[-5:]).to_dict()

print("ALS ready. users:", len(u2als), "items:", len(e2als))


In [ ]:

# CELL 5 — Candidate generation (memory-safe, chunked users)

def build_profiles(pos_df, editions_df, book_genres_df):
    e2b = editions_df[["edition_id", "book_id"]].drop_duplicates()
    e2a = editions_df[["edition_id", "author_id"]].drop_duplicates()

    ug = (pos_df.merge(e2b, on="edition_id", how="left")
            .merge(book_genres_df[["book_id", "genre_id"]], on="book_id", how="left")
            .dropna(subset=["genre_id"]))
    ug = ug.groupby(["user_id", "genre_id"]).size().rename("cnt").reset_index()
    ug["user_genre_affinity"] = ug["cnt"] / ug.groupby("user_id")["cnt"].transform("sum")

    ua = (pos_df.merge(e2a, on="edition_id", how="left")
            .dropna(subset=["author_id"]))
    ua = ua.groupby(["user_id", "author_id"]).size().rename("cnt").reset_index()
    ua["user_author_affinity"] = ua["cnt"] / ua.groupby("user_id")["cnt"].transform("sum")

    return ug[["user_id", "genre_id", "user_genre_affinity"]], ua[["user_id", "author_id", "user_author_affinity"]]


def generate_candidates(user_ids, item_feat, ug, ua, chunk_size=400):
    global_top = item_feat.sort_values("pop_log", ascending=False)["edition_id"].head(200).tolist()

    tmp_gen = item_feat[["edition_id", "book_id", "pop_log"]].merge(book_genres[["book_id", "genre_id"]], on="book_id", how="left")
    genre_index = tmp_gen.sort_values(["genre_id", "pop_log"], ascending=[True, False]).groupby("genre_id")["edition_id"].apply(lambda x: x.head(40).tolist()).to_dict()
    author_index = item_feat.sort_values(["author_id", "pop_log"], ascending=[True, False]).groupby("author_id")["edition_id"].apply(lambda x: x.head(30).tolist()).to_dict()

    ug_map = ug.sort_values(["user_id", "user_genre_affinity"], ascending=[True, False]).groupby("user_id")["genre_id"].apply(list).to_dict()
    ua_map = ua.sort_values(["user_id", "user_author_affinity"], ascending=[True, False]).groupby("user_id")["author_id"].apply(list).to_dict()

    out = []
    user_ids = list(user_ids)
    for st in range(0, len(user_ids), chunk_size):
        batch = user_ids[st:st + chunk_size]
        for uid in batch:
            cand = {}

            def add(eid, score, src):
                if (uid, eid) in seen_pairs:
                    return
                if eid not in cand:
                    cand[eid] = [0.0, 0, 0, 0, 0, 0]
                cand[eid][0] += score
                cand[eid][src] = 1

            # ALS U2I
            if uid in u2als:
                rec_ids, rec_scores = als.recommend(
                    userid=u2als[uid],
                    user_items=X[u2als[uid]],
                    N=120,
                    filter_already_liked_items=True,
                )
                for r, idx in enumerate(rec_ids):
                    add(int(als_items[idx]), 4.0 / (r + 1), 4)

            # ALS I2I from recent history
            for heid in user_hist.get(uid, []):
                if heid not in e2als:
                    continue
                ids, sims = als.similar_items(e2als[heid], N=16)
                for r, (idx, sim) in enumerate(zip(ids, sims)):
                    eid = int(als_items[idx])
                    if eid != heid:
                        add(eid, (3.0 * float(sim)) / (r + 1), 5)

            # genre / author / global
            for rank_g, gid in enumerate(ug_map.get(uid, [])[:10]):
                for eid in genre_index.get(gid, []):
                    add(int(eid), 1.5 / (rank_g + 1), 1)
            for rank_a, aid in enumerate(ua_map.get(uid, [])[:10]):
                for eid in author_index.get(aid, []):
                    add(int(eid), 1.5 / (rank_a + 1), 2)
            for r, eid in enumerate(global_top):
                add(int(eid), 0.5 / (r + 1), 3)

            top = sorted(cand.items(), key=lambda kv: kv[1][0], reverse=True)[:CANDS_PER_USER]
            for eid, info in top:
                out.append((uid, eid, info[0], info[1], info[2], info[3], info[4], info[5]))

        if (st // chunk_size) % 5 == 0:
            gc.collect()

    cands = pd.DataFrame(out, columns=[
        "user_id", "edition_id", "cand_score", "src_genre", "src_author", "src_global", "src_als_u2i", "src_als_i2i"
    ])
    for c in ["cand_score"]:
        cands[c] = cands[c].astype("float32")
    for c in ["src_genre", "src_author", "src_global", "src_als_u2i", "src_als_i2i"]:
        cands[c] = cands[c].astype("int8")

    # strict dedup
    cands = cands.sort_values(["user_id", "cand_score"], ascending=[True, False])
    cands = cands.drop_duplicates(["user_id", "edition_id"], keep="first")
    return cands

ug, ua = build_profiles(pos, editions, book_genres)
train_users = valid_log["user_id"].drop_duplicates().tolist()
train_cands = generate_candidates(train_users, item_features, ug, ua)
mem(train_cands, "train_cands")


In [ ]:

# CELL 6 — Ranking dataset + labels
valid_pos = valid_log[valid_log["event_type"].isin(POS_EVENTS)][["user_id", "edition_id"]].drop_duplicates()
valid_pos["target"] = 1

train_ds = train_cands.merge(valid_pos, on=["user_id", "edition_id"], how="left")
train_ds["target"] = train_ds["target"].fillna(0).astype("int8")

train_ds = train_ds.merge(user_features, on="user_id", how="left")
train_ds = train_ds.merge(item_features, on="edition_id", how="left")

# add affinities via compact merge maps
ug_key = ug.copy()
ua_key = ua.copy()
if "book_id" in item_features.columns:
    user_genre_aff = ug_key.groupby("user_id")["user_genre_affinity"].max().rename("user_genre_affinity_max")
    user_author_aff = ua_key.groupby("user_id")["user_author_affinity"].max().rename("user_author_affinity_max")
    train_ds = train_ds.merge(user_genre_aff.reset_index(), on="user_id", how="left")
    train_ds = train_ds.merge(user_author_aff.reset_index(), on="user_id", how="left")

for c in train_ds.columns:
    if train_ds[c].dtype == "float64":
        train_ds[c] = train_ds[c].astype("float32")
train_ds = train_ds.fillna(0)

mem(train_ds, "train_ds")
print("positive rate:", train_ds["target"].mean())


In [ ]:

# CELL 7 — CatBoostRanker GPU
feature_cols = [
    "cand_score", "src_genre", "src_author", "src_global", "src_als_u2i", "src_als_i2i",
    "user_activity", "user_uniq_editions", "user_activity_log",
    "edition_pop", "edition_recent_pop", "edition_unique_users", "read_pop", "wish_pop",
    "pop_log", "recent_pop_log", "read_ratio", "wish_ratio",
    "publication_year", "language_id", "publisher_id", "author_id",
]
for extra in ["gender", "age", "user_genre_affinity_max", "user_author_affinity_max"]:
    if extra in train_ds.columns:
        feature_cols.append(extra)
feature_cols = [c for c in feature_cols if c in train_ds.columns]

train_ds = train_ds.sort_values("user_id").reset_index(drop=True)
train_pool = Pool(
    data=train_ds[feature_cols],
    label=train_ds["target"],
    group_id=train_ds["user_id"],
)

ranker = CatBoostRanker(
    loss_function="YetiRank",
    eval_metric="NDCG:top=20",
    iterations=1400,
    learning_rate=0.04,
    depth=8,
    random_seed=SEED,
    task_type="GPU",
    verbose=200,
)
ranker.fit(train_pool)


In [ ]:

# CELL 8 — Full fit and inference for targets
full_pos = interactions[interactions["event_type"].isin(POS_EVENTS)].copy()

# retrain ALS on full log
full_pos["base_w"] = full_pos["event_type"].map({1: 1.0, 2: 2.0}).astype("float32")
mx = full_pos["event_ts"].max()
full_pos["days_old"] = (mx - full_pos["event_ts"]).dt.days.clip(lower=0)
full_pos["w"] = (full_pos["base_w"] * np.exp(-np.log(2) * full_pos["days_old"] / 30)).astype("float32")

u_enc_f = LabelEncoder(); i_enc_f = LabelEncoder()
u_idx_f = u_enc_f.fit_transform(full_pos["user_id"])
i_idx_f = i_enc_f.fit_transform(full_pos["edition_id"])
X_f = csr_matrix((full_pos["w"].to_numpy(), (u_idx_f, i_idx_f)), shape=(len(u_enc_f.classes_), len(i_enc_f.classes_)))

als_f = AlternatingLeastSquares(factors=ALS_FACTORS, iterations=ALS_ITERS, regularization=0.02, random_state=SEED, use_gpu=True)
als_f.fit(X_f)

u2als = {u: j for j, u in enumerate(u_enc_f.classes_)}
e2als = {e: j for j, e in enumerate(i_enc_f.classes_)}
als_items = i_enc_f.classes_
seen_pairs = set(zip(full_pos["user_id"].to_numpy(), full_pos["edition_id"].to_numpy()))
user_hist = full_pos.sort_values("event_ts").groupby("user_id")["edition_id"].apply(lambda s: list(s)[-5:]).to_dict()
X = X_f
als = als_f

# rebuild lightweight features on full
item_pop_f = full_pos.groupby("edition_id").size().astype("float32").rename("edition_pop")
item_recent_f = full_pos[full_pos["event_ts"] >= (mx - pd.Timedelta(days=31))].groupby("edition_id").size().astype("float32").rename("edition_recent_pop")
item_users_f = full_pos.groupby("edition_id")["user_id"].nunique().astype("float32").rename("edition_unique_users")
item_read_f = full_pos[full_pos["event_type"].eq(2)].groupby("edition_id").size().astype("float32").rename("read_pop")
item_wish_f = full_pos[full_pos["event_type"].eq(1)].groupby("edition_id").size().astype("float32").rename("wish_pop")
item_features_f = editions[["edition_id", "book_id", "author_id", "publication_year", "language_id", "publisher_id"]].copy()
for s in [item_pop_f, item_recent_f, item_users_f, item_read_f, item_wish_f]:
    item_features_f = item_features_f.merge(s.reset_index(), on="edition_id", how="left")
for c in ["edition_pop", "edition_recent_pop", "edition_unique_users", "read_pop", "wish_pop"]:
    item_features_f[c] = item_features_f[c].fillna(0).astype("float32")
item_features_f["pop_log"] = np.log1p(item_features_f["edition_pop"]).astype("float32")
item_features_f["recent_pop_log"] = np.log1p(item_features_f["edition_recent_pop"]).astype("float32")
item_features_f["read_ratio"] = (item_features_f["read_pop"]/(item_features_f["edition_pop"]+1)).astype("float32")
item_features_f["wish_ratio"] = (item_features_f["wish_pop"]/(item_features_f["edition_pop"]+1)).astype("float32")

user_act_f = full_pos.groupby("user_id").size().astype("float32").rename("user_activity")
user_uniq_f = full_pos.groupby("user_id")["edition_id"].nunique().astype("float32").rename("user_uniq_editions")
user_features_f = pd.DataFrame({"user_id": pd.Index(np.union1d(users["user_id"].unique(), targets["user_id"].unique()))})
user_features_f = user_features_f.merge(user_act_f.reset_index(), on="user_id", how="left")
user_features_f = user_features_f.merge(user_uniq_f.reset_index(), on="user_id", how="left")
for c in ["user_activity", "user_uniq_editions"]:
    user_features_f[c] = user_features_f[c].fillna(0).astype("float32")
user_features_f["user_activity_log"] = np.log1p(user_features_f["user_activity"]).astype("float32")
for col in ["gender", "age"]:
    if col in users.columns:
        user_features_f = user_features_f.merge(users[["user_id", col]], on="user_id", how="left")
        user_features_f[col] = pd.to_numeric(user_features_f[col], errors="coerce").fillna(0).astype("float32")

ug_f, ua_f = build_profiles(full_pos, editions, book_genres)

test_cands = generate_candidates(targets["user_id"].tolist(), item_features_f, ug_f, ua_f)
test_ds = test_cands.merge(user_features_f, on="user_id", how="left").merge(item_features_f, on="edition_id", how="left").fillna(0)

pred = ranker.predict(Pool(test_ds[feature_cols]))
test_ds["pred"] = pred.astype("float32")


In [ ]:

# CELL 9 — Final submission with strict contract
submission = test_ds.sort_values(["user_id", "pred", "edition_id"], ascending=[True, False, True])
submission = submission.drop_duplicates(["user_id", "edition_id"], keep="first")
submission["rank"] = submission.groupby("user_id").cumcount() + 1
submission = submission[submission["rank"] <= TOP_K][["user_id", "edition_id", "rank"]].copy()

# fill missing ranks/users via global popularity
global_top = item_features_f.sort_values("pop_log", ascending=False)["edition_id"].tolist()
out_rows = []
for uid in targets["user_id"].tolist():
    g = submission[submission["user_id"].eq(uid)]
    used = set(g["edition_id"].tolist())
    rank = 1
    for _, row in g.sort_values("rank").iterrows():
        out_rows.append((int(uid), int(row.edition_id), rank))
        rank += 1
    if rank <= TOP_K:
        for eid in global_top:
            if rank > TOP_K:
                break
            if eid in used or (uid, eid) in seen_pairs:
                continue
            out_rows.append((int(uid), int(eid), rank))
            used.add(eid)
            rank += 1

submission_final = pd.DataFrame(out_rows, columns=["user_id", "edition_id", "rank"])
submission_final = submission_final.sort_values(["user_id", "rank"]).reset_index(drop=True)

# contract checks
assert submission_final.groupby("user_id").size().eq(TOP_K).all()
assert not submission_final.duplicated(["user_id", "edition_id"]).any()
assert submission_final["rank"].between(1, TOP_K).all()

submission_final.to_csv("submission.csv", index=False)
print("Saved submission.csv", submission_final.shape)
submission_final.head(20)
